#  STEAM GAMES - NOTEBOOK 2
### What factors affect the popularity and sales of a video game?
#### Tool: Databricks + PySpark
### Dataset: s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json

#### This notebook is standalone. It reloads and rebuilds all dataframes
#### needed for the popularity analysis (no dependency on Notebook 1).


# View notebook at:
#### https://dbc-dc9ab534-89e4.cloud.databricks.com/editor/notebooks/2837560666121411?o=4359258060611116

#### CELL 1 - Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SteamPopularity").getOrCreate()
print("Spark version:", spark.version)


Spark version: 4.1.0


#### CELL 2 - Load and flatten the dataset

In [0]:
# Load the raw JSON from S3
raw_df = spark.read.option("multiLine", True).json(
    "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
)

# Flatten the nested 'data' struct into a usable dataframe
df = raw_df.select(
    F.col("data.appid").cast(IntegerType()).alias("app_id"),
    F.col("data.name").alias("game_name"),
    F.col("data.developer").alias("developer"),
    F.col("data.publisher").alias("publisher"),
    F.col("data.type").alias("game_type"),
    F.col("data.release_date").alias("release_date_raw"),
    F.col("data.price").alias("price_str"),
    F.col("data.discount").alias("discount_str"),
    F.col("data.positive").alias("positive_reviews"),
    F.col("data.negative").alias("negative_reviews"),
    F.col("data.required_age").alias("required_age_str"),
    F.col("data.platforms.windows").alias("on_windows"),
    F.col("data.platforms.mac").alias("on_mac"),
    F.col("data.platforms.linux").alias("on_linux"),
    F.col("data.genre").alias("genre"),
    F.col("data.languages").alias("supported_languages"),
)

# Price: remove non-numeric characters, convert from cents to dollars
df = df.withColumn(
    "price",
    F.when((F.col("price_str").isNull()) | (F.col("price_str") == ""), 0.0)
     .otherwise(
         F.round(
             F.regexp_replace(F.col("price_str"), r"[^0-9.]", "").cast(FloatType()) / 100,
             2
         )
     )
)

# Discount: same cleaning approach as price
df = df.withColumn(
    "discount_pct",
    F.when((F.col("discount_str").isNull()) | (F.col("discount_str") == ""), 0.0)
     .otherwise(F.regexp_replace(F.col("discount_str"), r"[^0-9.]", "").cast(FloatType()))
)

# Required age: strip non-numeric characters then cast
df = df.withColumn(
    "required_age",
    F.when((F.col("required_age_str").isNull()) | (F.col("required_age_str") == ""), 0)
     .otherwise(F.regexp_replace(F.col("required_age_str"), r"[^0-9]", "").cast(IntegerType()))
)

# Release year: extract 4-digit year from the date string
df = df.withColumn(
    "release_year",
    F.when((F.col("release_date_raw").isNull()) | (F.col("release_date_raw") == ""), None)
     .otherwise(F.regexp_extract(F.col("release_date_raw"), r"(\d{4})", 1).cast(IntegerType()))
)

# Free / discounted flags
df = df.withColumn("is_free",
    F.when((F.col("price").isNull()) | (F.col("price") == 0), True).otherwise(False)
)
df = df.withColumn("has_discount",
    F.when(F.col("discount_pct") > 0, True).otherwise(False)
)

print("Rows loaded:", df.count())
df.printSchema()


Rows loaded: 55691
root
 |-- app_id: integer (nullable = true)
 |-- game_name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- game_type: string (nullable = true)
 |-- release_date_raw: string (nullable = true)
 |-- price_str: string (nullable = true)
 |-- discount_str: string (nullable = true)
 |-- positive_reviews: long (nullable = true)
 |-- negative_reviews: long (nullable = true)
 |-- required_age_str: string (nullable = true)
 |-- on_windows: boolean (nullable = true)
 |-- on_mac: boolean (nullable = true)
 |-- on_linux: boolean (nullable = true)
 |-- genre: string (nullable = true)
 |-- supported_languages: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- required_age: integer (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- is_free: boolean (nullable = false)
 |-- has_discount: boolean (nullable = false)



#### CELL 3 - Build all metrics: popularity, quality, estimated sales and revenue

In [0]:
# --- Popularity and quality metrics ---

# total_reviews = positive + negative (proxy for number of players)
df = df.withColumn(
    "total_reviews",
    F.col("positive_reviews") + F.col("negative_reviews")
)

# positive_ratio = share of positive reviews (proxy for player satisfaction)
df = df.withColumn(
    "positive_ratio",
    F.when(
        F.col("total_reviews").isNotNull() & (F.col("total_reviews") > 0),
        F.round(F.col("positive_reviews") / F.col("total_reviews"), 4)
    ).otherwise(None)
)

# success_score combines volume and quality into one metric
# a game with many reviews AND a high positive ratio scores highest
df = df.withColumn(
    "success_score",
    F.when(
        F.col("positive_ratio").isNotNull() & F.col("total_reviews").isNotNull(),
        F.round(F.col("total_reviews") * F.col("positive_ratio"), 0)
    ).otherwise(None)
)

# --- Estimated sales using the Boxleiter method ---
# Steam does not publish official sales data.
# The Boxleiter method estimates: sales ≈ total_reviews × multiplier
# Multiplier range used in the industry: 30 (conservative) to 77 (optimistic)
# Source: https://newsletter.gamediscover.co/p/how-that-game-sold-on-steam-using

BOXLEITER_LOW  = 30
BOXLEITER_HIGH = 77

df = df.withColumn(
    "est_sales_low",
    F.when(
        F.col("total_reviews").isNotNull() & (F.col("total_reviews") > 0),
        F.col("total_reviews") * BOXLEITER_LOW
    ).otherwise(None)
).withColumn(
    "est_sales_high",
    F.when(
        F.col("total_reviews").isNotNull() & (F.col("total_reviews") > 0),
        F.col("total_reviews") * BOXLEITER_HIGH
    ).otherwise(None)
)

# Estimated revenue = estimated sales × price
# Free games have zero revenue despite potentially high sales volume
df = df.withColumn(
    "est_revenue_low",
    F.when(
        F.col("est_sales_low").isNotNull() & (F.col("price") > 0),
        F.round(F.col("est_sales_low") * F.col("price"), 0)
    ).otherwise(None)
).withColumn(
    "est_revenue_high",
    F.when(
        F.col("est_sales_high").isNotNull() & (F.col("price") > 0),
        F.round(F.col("est_sales_high") * F.col("price"), 0)
    ).otherwise(None)
)

# --- Overview ---
print("Popularity and sales metrics overview:")
df.select(
    "total_reviews", "positive_ratio", "success_score",
    "est_sales_high", "est_revenue_high"
).describe().show()

print("Games with 0 or null reviews:",
      df.filter((F.col("total_reviews") == 0) | F.col("total_reviews").isNull()).count())
print("Games with at least 10 reviews:",
      df.filter(F.col("total_reviews") >= 10).count())
print("Games with at least 100 reviews:",
      df.filter(F.col("total_reviews") >= 100).count())


Popularity and sales metrics overview:
+-------+------------------+-------------------+------------------+------------------+-------------------+
|summary|     total_reviews|     positive_ratio|     success_score|    est_sales_high|   est_revenue_high|
+-------+------------------+-------------------+------------------+------------------+-------------------+
|  count|             55691|              55528|             55528|             55528|              47762|
|   mean|1712.7132929916863| 0.7365361763434721|1475.1972158190463|132266.04833597466|  2610728.426238432|
| stddev| 35687.61118456136|0.24231825811705743| 31029.04986197961|2751967.1123686205|3.856984507681988E7|
|    min|                 0|                0.0|               0.0|                77|               22.0|
|    max|           6730438|                1.0|         5943650.0|         518243726|      3.330285968E9|
+-------+------------------+-------------------+------------------+------------------+-------------------

#### CELL 4 - Build helper dataframes

In [0]:
# --- Genres dataframe ---
# The 'genre' column is a comma-separated string e.g. "Action,Indie,RPG"
# We split and explode it to get one row per genre per game
genres_df = (
    df.select(
        "app_id", "game_name", "publisher", "price",
        "release_year", "positive_reviews", "negative_reviews",
        "total_reviews", "positive_ratio", "success_score",
        "est_sales_low", "est_sales_high",
        "est_revenue_low", "est_revenue_high",
        F.explode(F.split(F.col("genre"), ",")).alias("genre_raw")
    )
    .withColumn("genre", F.trim(F.col("genre_raw")))
    .drop("genre_raw")
    .filter(F.col("genre") != "")
)

# --- Languages dataframe ---
# The 'supported_languages' column is a comma-separated string
# Some entries contain HTML tags like <strong>*</strong> that we clean up
languages_df = (
    df.select(
        "app_id",
        F.explode(F.split(F.col("supported_languages"), ",")).alias("language_raw")
    )
    .withColumn(
        "language",
        F.trim(F.regexp_replace(F.col("language_raw"), r"<[^>]+>|\*", ""))
    )
    .drop("language_raw")
    .filter(F.col("language") != "")
)

# Language count per game (used in factor 5 analysis)
df_lang_count = (
    languages_df.groupBy("app_id")
    .agg(F.count("language").alias("num_languages"))
)
df = df.join(df_lang_count, on="app_id", how="left")

print("Genres dataframe sample:")
genres_df.show(5, truncate=False)

print("Languages dataframe sample:")
languages_df.show(5, truncate=False)


Genres dataframe sample:
+-------+--------------+------------------------+-----+------------+----------------+----------------+-------------+--------------+-------------+-------------+--------------+---------------+----------------+---------+
|app_id |game_name     |publisher               |price|release_year|positive_reviews|negative_reviews|total_reviews|positive_ratio|success_score|est_sales_low|est_sales_high|est_revenue_low|est_revenue_high|genre    |
+-------+--------------+------------------------+-----+------------+----------------+----------------+-------------+--------------+-------------+-------------+--------------+---------------+----------------+---------+
|10     |Counter-Strike|Valve                   |9.99 |2000        |201215          |5199            |206414       |0.9748        |201212.0     |6192420      |15893878      |6.1862276E7    |1.58779841E8    |Action   |
|1000000|ASCENXION     |PsychoFlux Entertainment|9.99 |2021        |27              |5               |3

####  CELL 5 - Top games by estimated sales and revenue

In [0]:
# Top games by estimated sales volume (most units sold)
print("Top 20 games by estimated sales:")
top_sales = (
    df.filter(F.col("est_sales_high").isNotNull())
      .select(
          "game_name",
          "publisher",
          "est_sales_high"
      )
      .orderBy(F.desc("est_sales_high"))
      .limit(20)
)

display(top_sales)

# Top games by estimated revenue (best commercial performers)
# Note: free games are excluded here because their revenue is 0
print("Top 20 games by estimated revenue:")
top_revenue = (
    df.filter(F.col("est_revenue_high").isNotNull())
      .select(
          "game_name",
          "publisher",
          "est_revenue_high"
      )
      .orderBy(F.desc("est_revenue_high"))
      .limit(20)
)

display(top_revenue)


Top 20 games by estimated sales:


game_name,publisher,est_sales_high
Counter-Strike: Global Offensive,Valve,518243726
PUBG: BATTLEGROUNDS,"KRAFTON, Inc.",161228452
Dota 2,Valve,142666447
Grand Theft Auto V,Rockstar Games,111083588
Tom Clancy's Rainbow Six Siege,Ubisoft,83634089
Terraria,Re-Logic,79856007
Team Fortress 2,Valve,69594910
Garry's Mod,Valve,68625326
Rust,Facepunch Studios,65039359
Left 4 Dead 2,Valve,50871128


Databricks visualization. Run in Databricks to view.

Top 20 games by estimated revenue:


game_name,publisher,est_revenue_high
Grand Theft Auto V,Rockstar Games,3.330285968E9
Rust,Facepunch Studios,2.600923966E9
Cyberpunk 2077,CD PROJEKT RED,2.573414606E9
ELDEN RING,"FromSoftware Inc., Bandai Namco Entertainment",2.504874471E9
The Witcher 3: Wild Hunt,CD PROJEKT RED,2.025739199E9
Tom Clancy's Rainbow Six Siege,Ubisoft,1.671845439E9
DARK SOULS III,"FromSoftware, Inc., Bandai Namco Entertainment",1.559290375E9
Red Dead Redemption 2,Rockstar Games,1.425873155E9
No Man's Sky,Hello Games,1.15204982E9
DayZ,Bohemia Interactive,1.113666803E9


Databricks visualization. Run in Databricks to view.

### CELL 6 - Factor 1: Does PRICE affect popularity and sales?

In [0]:

# Price vs Popularity & Revenue
# Objective:
# - Identify which price ranges attract the largest audiences
# - Compare sales volume and revenue generation
# - Highlight the trade-off between popularity and monetization


price_vs_sales = (
    df.filter(F.col("est_sales_high").isNotNull())
    .withColumn(
        "price_range",
        F.when(F.col("price") == 0,   "00. Free")
         .when(F.col("price") <= 5,   "01. 0-5")
         .when(F.col("price") <= 10,  "02. 5-10")
         .when(F.col("price") <= 20,  "03. 10-20")
         .when(F.col("price") <= 30,  "04. 20-30")
         .when(F.col("price") <= 60,  "05. 30-60")
         .otherwise(                  "06. 60+")
    )
    .groupBy("price_range")
    .agg(
        F.count("app_id").alias("game_count"),
        F.expr("percentile_approx(est_sales_high, 0.5)").alias("median_est_sales"),
        F.avg("positive_ratio").alias("avg_positive_ratio"),
        F.expr(
            "percentile_approx(est_revenue_high, 0.5)"
        ).alias("median_est_revenue")
    )
    .orderBy("price_range")
)

price_vs_sales.show(10)
print("Distribution of games by price range")
display(
    price_vs_sales.select(
        "price_range",
        "game_count"
    )
)
print("Median estimated sales by price range")
display(
    price_vs_sales.select(
        "price_range",
        "median_est_sales"
    )
)
print("Median estimated revenue by price range")
display(
    price_vs_sales.select(
        "price_range",
        "median_est_revenue"
    )
)

+-----------+----------+----------------+------------------+------------------+
|price_range|game_count|median_est_sales|avg_positive_ratio|median_est_revenue|
+-----------+----------+----------------+------------------+------------------+
|   00. Free|      7766|            3619|0.6966465619366454|              NULL|
|    01. 0-5|     23403|            1155|0.7243046959791475|            2758.0|
|   02. 5-10|     12411|            1771|0.7543314962533213|           14615.0|
|  03. 10-20|      8994|            6160| 0.768360117856345|           96955.0|
|  04. 20-30|      1788|           24486|0.7684068232662186|          658131.0|
|  05. 30-60|      1044|           50820|0.7692438697318008|         2325120.0|
|    06. 60+|       122|            1001|0.7186860655737705|          107789.0|
+-----------+----------+----------------+------------------+------------------+

Distribution of games by price range


price_range,game_count
00. Free,7766
01. 0-5,23403
02. 5-10,12411
03. 10-20,8994
04. 20-30,1788
05. 30-60,1044
06. 60+,122


Databricks visualization. Run in Databricks to view.

Median estimated sales by price range


price_range,median_est_sales
00. Free,3619
01. 0-5,1155
02. 5-10,1771
03. 10-20,6160
04. 20-30,24486
05. 30-60,50820
06. 60+,1001


Databricks visualization. Run in Databricks to view.

Median estimated revenue by price range


price_range,median_est_revenue
00. Free,null
01. 0-5,2758.0
02. 5-10,14615.0
03. 10-20,96955.0
04. 20-30,658131.0
05. 30-60,2325120.0
06. 60+,107789.0


Databricks visualization. Run in Databricks to view.

#### CELL 7 - Factor 2: Does GENRE affect popularity and sales?

In [0]:
genre_vs_sales = (
    genres_df.filter(F.col("total_reviews").isNotNull())
    .groupBy("genre")
    .agg(
        F.count("app_id").alias("game_count"),
        F.expr("percentile_approx(est_sales_high, 0.5)").alias("median_est_sales"),
        F.sum("est_sales_high").alias("total_est_sales"),
        F.expr("percentile_approx(est_revenue_high, 0.5)").alias("median_est_revenue"),
        F.sum("est_revenue_high").alias("total_est_revenue"),
        F.avg("positive_ratio").alias("avg_positive_ratio"),
    )
    .filter(F.col("game_count") >= 20)
    .orderBy(F.desc("median_est_sales"))
)

print("Genre vs estimated sales and revenue:")
print("1. Genre vs Sales")
display(genre_vs_sales)
print("2. Genre vs Revenue")
display(
    genre_vs_sales.select("genre", "median_est_revenue"
                          )
    )


# Insight: the genre with the most total sales is not necessarily the most profitable per game



Genre vs estimated sales and revenue:
1. Genre vs Sales


genre,game_count,median_est_sales,total_est_sales,median_est_revenue,total_est_revenue,avg_positive_ratio
Free to Play,3393,10241,1771235081,7598.0,3.1736248E7,0.7218509149940988
Massively Multiplayer,1460,9548,840665826,17205.0,8.939488874E9,0.6290803424657537
Nudity,45,6391,806575,20839.0,4810115.0,0.6712688888888888
Gore,99,3465,2700313,8837.0,1.1958172E7,0.6319939393939392
RPG,9534,3311,1747888912,20000.0,3.7914428488E10,0.730576722689073
Web Publishing,89,2772,3020094,57392.0,9.4600683E7,0.7196707865168543
Simulation,10836,2772,1383913454,16898.0,2.6231509499E10,0.6936708576186494
Sexual Content,54,2772,594671,11824.0,4565085.0,0.6760722222222221
Strategy,10895,2618,1216314715,14044.0,2.3322210212E10,0.7193268205222507
Violent,168,2464,4045580,8302.0,1.7639585E7,0.5913185628742517


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

2. Genre vs Revenue


genre,median_est_revenue
Free to Play,7598.0
Massively Multiplayer,17205.0
Nudity,20839.0
Gore,8837.0
RPG,20000.0
Web Publishing,57392.0
Simulation,16898.0
Sexual Content,11824.0
Strategy,14044.0
Violent,8302.0


Databricks visualization. Run in Databricks to view.

#### CELL 8 - Factor 3: Does PLATFORM AVAILABILITY affect popularity and sales?

In [0]:
platform_vs_sales = (
    df.filter(F.col("total_reviews").isNotNull())
    .withColumn(
        "num_platforms",
        F.col("on_windows").cast(IntegerType()) +
        F.col("on_mac").cast(IntegerType()) +
        F.col("on_linux").cast(IntegerType())
    )
    .groupBy("num_platforms")
    .agg(
        F.count("app_id").alias("game_count"),
        F.expr("percentile_approx(est_sales_high, 0.5)").alias("median_est_sales"),
        F.expr("percentile_approx(est_revenue_high, 0.5)").alias("median_est_revenue"),
        F.avg("positive_ratio").alias("avg_positive_ratio"),
    )
    .orderBy("num_platforms")
)
print("Number of platforms vs estimated sales:")
display(platform_vs_sales)


# Hypothesis: multi-platform games reach a wider audience

Number of platforms vs estimated sales:


num_platforms,game_count,median_est_sales,median_est_revenue,avg_positive_ratio
1,41285,1617,7374.0,0.7221516959035926
2,7599,2849,15232.0,0.7729154221635857
3,6807,6160,32566.0,0.7831167010309239


Databricks visualization. Run in Databricks to view.

#### CELL 9 - Factor 4: Does RELEASE YEAR affect popularity and sales?

In [0]:
# Important bias: older games have had more time to accumulate reviews.
# We note this limitation but include the analysis for context.

year_vs_sales = (
    df.filter(
        F.col("total_reviews").isNotNull() &
        F.col("release_year").isNotNull() &
        (F.col("release_year") >= 2000) &
        (F.col("release_year") <= 2024)
    )
    .groupBy("release_year")
    .agg(
        F.count("app_id").alias("game_count"),
        F.expr("percentile_approx(est_sales_high, 0.5)").alias("median_est_sales"),
        F.expr("percentile_approx(est_revenue_high, 0.5)").alias("median_est_revenue"),
        F.avg("positive_ratio").alias("avg_positive_ratio"),
    )
    .orderBy("release_year")
)
print("Release year vs estimated sales:")
display(year_vs_sales)


# Note: the decline in recent years reflects the time bias, not lower quality

Release year vs estimated sales:


release_year,game_count,median_est_sales,median_est_revenue,avg_positive_ratio
2000,2,363055,1811644.0,0.89665
2001,4,184646,921384.0,0.895675
2002,1,778547,1.167042E7,0.8706
2003,3,63140,109864.0,0.8332
2004,6,902132,9012299.0,0.9079833333333333
2005,6,72380,723076.0,0.8268166666666668
2006,61,44198,377433.0,0.8467508196721313
2007,98,72996,907935.0,0.8082010204081636
2008,159,33341,184615.0,0.7762503144654084
2009,311,33341,221538.0,0.7666556634304207


Databricks visualization. Run in Databricks to view.

####  CELL 10 - Factor 5: Does NUMBER OF SUPPORTED LANGUAGES affect popularity?

In [0]:
# More languages = wider potential audience = more players = more reviews/sales

lang_vs_sales = (
    df.filter(
        F.col("total_reviews").isNotNull() &
        F.col("num_languages").isNotNull()
    )
    .withColumn(
        "lang_bucket",
        F.when(F.col("num_languages") == 1,  "1 - Only 1 language")
         .when(F.col("num_languages") <= 3,  "2 - 2-3 languages")
         .when(F.col("num_languages") <= 5,  "3 - 4-5 languages")
         .when(F.col("num_languages") <= 10, "4 - 6-10 languages")
         .otherwise(                         "5 - 10+ languages")
    )
    .groupBy("lang_bucket")
    .agg(
        F.count("app_id").alias("game_count"),
        F.expr("percentile_approx(est_sales_high, 0.5)").alias("median_est_sales"),
        F.expr("percentile_approx(est_revenue_high, 0.5)").alias("median_est_revenue"),
        F.avg("positive_ratio").alias("avg_positive_ratio"),
    )
    .orderBy("lang_bucket")
)

print("Number of supported languages vs estimated sales:")
display(lang_vs_sales)

# → Databricks: bar chart | X = lang_bucket | Y = avg_est_sales


Number of supported languages vs estimated sales:


lang_bucket,game_count,median_est_sales,median_est_revenue,avg_positive_ratio
1 - Only 1 language,29655,1232,4995.0,0.7240645155832267
2 - 2-3 languages,10966,1771,8288.0,0.7363878028709893
3 - 4-5 languages,4096,5775,44290.0,0.7509752446183979
4 - 6-10 languages,6384,11858,108947.0,0.7609160577073911
5 - 10+ languages,4579,11242,66187.0,0.7711592641261508


Databricks visualization. Run in Databricks to view.

#### CELL 11 - Factor 6: Does AGE RESTRICTION affect popularity and sales?

In [0]:
age_vs_sales = (
    df.filter(F.col("total_reviews").isNotNull())
    .withColumn(
        "age_group",
        F.when(F.col("required_age") == 0,  "All ages")
         .when(F.col("required_age") < 16,  "Under 16")
         .when(F.col("required_age") == 16, "16+")
         .otherwise(                        "18+")
    )
    .groupBy("age_group")
    .agg(
        F.count("app_id").alias("game_count"),
        F.expr("percentile_approx(est_sales_high, 0.5)").alias("median_est_sales"),
        F.expr("percentile_approx(est_revenue_high, 0.5)").alias("median_est_revenue"),
        F.avg("positive_ratio").alias("avg_positive_ratio"),
    )
    .orderBy(F.desc("median_est_sales"))
)
print("Age restriction vs estimated sales:")
display(age_vs_sales)

# → Databricks: bar chart | X = age_group | Y = avg_est_sales


Age restriction vs estimated sales:


age_group,game_count,median_est_sales,median_est_revenue,avg_positive_ratio
Under 16,355,196427,3403951.0,0.744240563380282
16+,38,20482,101023.0,0.6617921052631579
18+,268,17633,227105.0,0.713416479400749
All ages,55030,2002,9238.0,0.736650599620913


Databricks visualization. Run in Databricks to view.

%md

## Global Conclusion — Steam Market Analysis for Ubisoft

This project analyzed the Steam games catalog using PySpark on Databricks
across two notebooks: a global market overview (Notebook 1) and a focused
analysis of the factors driving popularity and sales (Notebook 2).

Since Steam does not publish official sales data, two estimation proxies
were used throughout:
- **median_est_sales** = `total_reviews × 77` (Boxleiter method, optimistic estimate)
- **median_est_revenue** = `median_est_sales × price`

Median values were preferred over averages to limit the distortion caused
by outlier titles.

---

## What the Steam Market Looks Like

The Steam catalog is vast and highly competitive. A small number of publishers
dominate in volume, while the majority of developers have released only one
or two titles. Game releases grew steadily through the 2010s before
stabilizing around 2018–2019. The Covid period (2020–2021) did not
significantly reduce release activity.

Action, Indie and RPG are the three most represented genres. Windows is the
dominant platform by a wide margin. English is the default language, but
supporting additional languages is strongly correlated with commercial
performance. Discounts are widely used as a standard visibility strategy
across the catalog.

---

## What Drives Popularity and Sales

### 1. Genre

Free to Play and Massively Multiplayer games lead in median estimated sales.
Among premium genres, RPG generates the highest median revenue (20,000),
followed by Massively Multiplayer (17,205). Action and Indie have massive
total sales but low medians, reflecting a large volume of small titles
pulling performance down.

### 2. Price

The $30–60 range peaks in both median sales (50,820) and median revenue
(2,325,120) — nearly 24x more revenue than the $10–20 range (96,955).
Free games have moderate median sales (3,619) but generate no direct revenue.
The $60+ tier collapses in both metrics, confirming that Steam players
strongly resist console-style pricing.

### 3. Platform Availability

A clear and consistent progression: games on 3 platforms achieve median
sales 3.8x higher than single-platform games (6,160 vs 1,617) and median
revenue 4.4x higher (32,566 vs 7,374). Multi-platform releases also show
higher player satisfaction (avg positive ratio 0.783 vs 0.722).

### 4. Language Support

Performance increases steadily with language count, peaking at 6–10
languages (median sales: 11,858 — median revenue: 108,947). Beyond 10
languages, median revenue drops (66,187), likely because heavily localised
titles tend to be free-to-play. Single-language games show the weakest
performance across all metrics.

### 5. Release Year

Pre-2014 games show very high medians due to years of review accumulation.
A steady decline follows after 2014, driven by increased market saturation.
The average positive ratio has been rising since 2020, reaching 0.794 in
2022 — recent games are better received despite lower sales volumes.

### 6. Age Restriction

Counter-intuitively, games rated "Under 16" show the highest median sales
(196,427) and revenue (3,403,951), driven by major AAA titles with
Teen/PEGI 12–15 ratings. "All ages" games are the most numerous (55,030)
but have the lowest median sales (2,002), reflecting the mass of small
indie titles. Age rating reflects game type more than commercial potential.

---

## Strategic Recommendations for Ubisoft

| Priority | Recommendation |
|---|---|
| 1 | **Genre**: target RPG or Massively Multiplayer for the best median revenue per game |
| 2 | **Price**: price between $30–60 — this range maximises both sales and revenue on Steam |
| 3 | **Platforms**: release on all 3 platforms — median sales are 3.8x higher |
| 4 | **Languages**: support 6–10 languages — the sweet spot for revenue reach |
| 5 | **Quality**: invest in post-launch support — player satisfaction is rising for recent titles |

---

## Limitations

- All sales and revenue figures are estimates based on the Boxleiter method,
  not official Steam data.
- The analysis identifies correlations, not causal relationships.
- External factors such as marketing spend, press coverage and streamer
  visibility are absent from the dataset but likely have significant impact.
- A predictive model (e.g., regression analysis) would be required to
  quantify the individual contribution of each factor more precisely.

---

*Analysis conducted with PySpark on Databricks | Dataset: Steam (Jedha S3 bucket)*